# Early and 2024 samples (superseded)

These notebooks produced the first dataset (earliest 2014 videos) and a 2024 sample.
They are kept for reference. The 2024 sample (`recent_60k.parquet`) will be reused as an
out-of-time check at the end of the project. The main dataset is built in `01c`.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

REPO = "datasocial/tiktok-5.6B-videos"
items = [i for i in HfApi().list_repo_tree(REPO, path_in_repo="data", repo_type="dataset")
         if i.path.endswith(".parquet")]
items = sorted(items, key=lambda i: i.path)
for i in items[-30:]:
    print(i.path, round(i.size / 1e9, 2), "GB")

In [ ]:
import random
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl
from huggingface_hub import HfFileSystem

ROOT = Path.cwd().parent
OUT = ROOT / "data" / "sample_recent"

MONTHS = [f"2024-{m:02d}" for m in range(1, 13)]   # change if needed
PER_MONTH = 5_000
SEED = 42

fs = HfFileSystem()
random.seed(SEED)
parts = []

for m in MONTHS:
    path = f"datasets/{REPO}/data/{m}.parquet"
    with fs.open(path, "rb") as fh:
        pf = pq.ParquetFile(fh)
        groups = list(range(pf.num_row_groups))
        random.shuffle(groups)
        chunks, got = [], 0
        for g in groups:
            t = pf.read_row_group(g)
            chunks.append(t)
            got += t.num_rows
            if got >= PER_MONTH and len(chunks) >= 3:
                break
        tbl = pa.concat_tables(chunks)
    d = pl.from_arrow(tbl).sample(n=min(PER_MONTH, tbl.num_rows), seed=SEED)
    parts.append(d)
    print(m, "row groups read:", len(chunks), "kept:", d.height)

recent = pl.concat(parts).sort("posted_at")
recent.write_parquet(OUT / "recent_60k.parquet")
print(recent.shape)
print(recent.select(["width", "height", "country", "language", "views"]).head(5))